# Praktikum 1.1: Scraping Data Gizi Open Food Facts

Notebook ini berisi tahapan scraping dan menyimpannya ke `data/`.

Endpoint: `https://world.openfoodfacts.org/api/v2/search`

In [9]:
import json
import os
import time
from pathlib import Path

import pandas as pd
import requests

API_URL = "https://world.openfoodfacts.org/api/v2/search"
USER_AGENT = f"tugas-datmin/1.0"

DIR_DATA = Path("data")
DIR_CACHE = DIR_DATA / "raw"
RESULT_FILE = DIR_DATA / "off_nutrition.csv"

# Kolom gizi per 100 g
NUTRIENTS = [
    "energy-kcal_100g",
    "fat_100g",
    "saturated-fat_100g",
    "carbohydrates_100g",
    "sugars_100g",
    "fiber_100g",
    "proteins_100g",
    "salt_100g",
    "fruits-vegetables-nuts-estimate-from-ingredients_100g",
]

META_FIELDS = [
    "code",
    "product_name",
    "brands",
    "nutriscore_grade",
    "nutriscore_score",
    "nova_group",
]

print("Folder data sudah ada:", DIR_DATA.exists())
print("CSV hasil sudah ada  :", RESULT_FILE.exists())

Folder data sudah ada: False
CSV hasil sudah ada  : False


## Ambil halaman

Fungsi pintar untuk scraping satu halaman (berisi ~100 items).

In [10]:
def get_page(
    grade: str,
    page: int,
    rows: int = 100,
    max_retries: int = 8,
    delay: float = 5.0,
) -> list[dict]:
    data = DIR_CACHE / f"off_{grade}_p{page}.json"
    if data.exists():  # sudah pernah didownload
        return json.loads(data.read_text())

    for it in range(1, max_retries + 1):
        respons = requests.get(
            API_URL,
            params={
                "nutrition_grades_tags": grade,
                "fields": ",".join(META_FIELDS + ["nutriments"]),
                "page": page,
                "page_size": rows,
                "json": 1,
            },
            headers={"User-Agent": USER_AGENT},
            timeout=120,
        )
        if respons.status_code == 200:
            product = respons.json().get("products", [])
            data.write_text(json.dumps(product))
            return product
        if respons.status_code not in (429, 500, 502, 503, 504):
            respons.raise_for_status()  # error permanen
        print(f"  HTTP {respons.status_code} (grade={grade}, halaman={page}), lanjut ke percobaan {it}/{max_retries}...")
        time.sleep(delay * it)

    raise RuntimeError(f"Gagal mengambil grade={grade} halaman={page}")

## Compile menjadi satu

Respons API menaruh semua angka gizi di dalam nested objects `nutriments`. `ratakan()` harus di-flatten sebelum untuk digunakan jadi `DataFrame`.

In [11]:
def flatten(product: dict) -> dict:
    nutrisi = product.get("nutriments") or {}
    rows = {col: product.get(col) for col in META_FIELDS}
    rows.update({col: nutrisi.get(col) for col in NUTRIENTS})
    return rows


def get_dataset(grades: str = "abcde", page: int = 3, delay: float = 5.0) -> pd.DataFrame:
    rows = []
    for grade in grades:
        for no in range(1, page + 1):
            product = get_page(grade, no, delay=delay)
            print(f"grade {grade} halaman {no}: {len(product)} produk")
            if not product:
                break  # halaman kosong: hasil untuk grade ini sudah habis
            rows.extend(flatten(p) for p in product)
            time.sleep(delay)
    return pd.DataFrame(rows)

## Jalankan Scraping

Kode utama untuk menjalankan fungsi secara teratur.

In [12]:
if RESULT_FILE.exists():
    df_mentah = pd.read_csv(RESULT_FILE, dtype={"code": str})
    print(f"Dibaca dari cache: {RESULT_FILE} ({len(df_mentah)} baris, 0 permintaan jaringan)")
else:
    print("Cache tidak ditemukan, panggil API Open Food Facts...")
    DIR_CACHE.mkdir(parents=True, exist_ok=True)
    df_mentah = get_dataset(page=4)
    df_mentah.to_csv(RESULT_FILE, index=False)
    print(f"\nTersimpan ke {RESULT_FILE}")

print("Ukuran:", df_mentah.shape)
df_mentah.head()

Cache tidak ditemukan, panggil API Open Food Facts...
  HTTP 503 (grade=a, halaman=1), lanjut ke percobaan 1/8...
grade a halaman 1: 100 produk
grade a halaman 2: 100 produk
  HTTP 503 (grade=a, halaman=3), lanjut ke percobaan 1/8...
  HTTP 503 (grade=a, halaman=3), lanjut ke percobaan 2/8...
grade a halaman 3: 100 produk
grade a halaman 4: 100 produk
  HTTP 503 (grade=b, halaman=1), lanjut ke percobaan 1/8...
  HTTP 503 (grade=b, halaman=1), lanjut ke percobaan 2/8...
  HTTP 503 (grade=b, halaman=1), lanjut ke percobaan 3/8...
  HTTP 503 (grade=b, halaman=1), lanjut ke percobaan 4/8...
grade b halaman 1: 100 produk
grade b halaman 2: 100 produk
grade b halaman 3: 100 produk
  HTTP 503 (grade=b, halaman=4), lanjut ke percobaan 1/8...
grade b halaman 4: 100 produk
  HTTP 503 (grade=c, halaman=1), lanjut ke percobaan 1/8...
grade c halaman 1: 100 produk
grade c halaman 2: 100 produk
grade c halaman 3: 100 produk
  HTTP 503 (grade=c, halaman=4), lanjut ke percobaan 1/8...
grade c halaman 

,code,product_name,brands,nutriscore_grade,nutriscore_score,nova_group,energy-kcal_100g,fat_100g,saturated-fat_100g,carbohydrates_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-nuts-estimate-from-ingredients_100g
0,6111035000430,Sidi Ali,سيدي علي,a,0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.00
1,6111242100992,Perly,Jaouda,a,0,3.0,97.0,3.0,0.0,9.4,9.4,0.0,8.0,0.00000,6.85
2,6111035000058,Eau minérale naturelle,sidi ali,a,0,1.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,0.00600,0.00
3,6111035002175,Sidi Ali,Sidi Ali,a,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00
4,3274080005003,isabelle,Cristaline,a,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00275,0.00


## Periksa hasil

Sebelum data digunakan EDA, cek dulu kondisinya.

In [14]:
print("Sebaran label Nutri-Score:")
print(df_mentah["nutriscore_grade"].value_counts().sort_index())

print("\nProporsi nilai terisi per kolom gizi:")
print(df_mentah[NUTRIENTS].notna().mean().sort_values(ascending=False).round(3))

print("\nData siap dianalisis di 2_eda.ipynb")

Sebaran label Nutri-Score:
nutriscore_grade
a    400
b    400
c    400
d    400
e    399
Name: count, dtype: int64

Proporsi nilai terisi per kolom gizi:
salt_100g                                                0.959
energy-kcal_100g                                         0.952
fat_100g                                                 0.949
proteins_100g                                            0.949
sugars_100g                                              0.948
saturated-fat_100g                                       0.948
carbohydrates_100g                                       0.946
fruits-vegetables-nuts-estimate-from-ingredients_100g    0.887
fiber_100g                                               0.733
dtype: float64

Data siap dianalisis di 2_eda.ipynb
